# Model 1: Clothing Classification
**Dataset:** Fashion Product Images (Kaggle) + iMaterialist Fashion  
**Architecture:** EfficientNet-B0 (transfer learning)  
**Output:** `clothing_classifier.pth`  
**Target:** Accuracy > 80%

---
## Setup
Run on **Kaggle Notebook** (free GPU T4) hoặc Google Colab.

```bash
# Kaggle: bật GPU trong Settings → Accelerator → GPU T4 x2
# Download dataset tại: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
```

In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install torch torchvision tqdm scikit-learn pandas pillow matplotlib --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# CELL 2: Download dataset from Kaggle
# ============================================================
import os

# Nếu chạy trên Kaggle notebook, dataset đã có sẵn tại /kaggle/input/
# Nếu chạy local/Colab: đặt kaggle.json vào ~/.kaggle/ rồi chạy lệnh dưới

DATASET_PATH = "/kaggle/input/fashion-product-images-small"  # Kaggle

if not os.path.exists(DATASET_PATH):
    print("Downloading dataset...")
    os.makedirs("./data", exist_ok=True)
    !kaggle datasets download paramaggarwal/fashion-product-images-small -p ./data --unzip

    # Auto-detect: file có thể được giải nén thẳng vào ./data/ hoặc vào subfolder
    candidate = "./data/fashion-product-images-small"
    if os.path.exists(candidate):
        DATASET_PATH = candidate
    elif os.path.exists("./data/styles.csv"):
        DATASET_PATH = "./data"
    else:
        # Tìm thư mục chứa styles.csv
        for root, dirs, files in os.walk("./data"):
            if "styles.csv" in files:
                DATASET_PATH = root
                break

print(f"Dataset path: {DATASET_PATH}")
print(os.listdir(DATASET_PATH))

In [ ]:
# ============================================================
# CELL 3: Explore dataset
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Load metadata
df = pd.read_csv(f"{DATASET_PATH}/styles.csv", on_bad_lines='skip')
print(f"Total samples: {len(df)}")
print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nArticleType distribution:")
print(df['articleType'].value_counts().head(20))

In [ ]:
# ============================================================
# CELL 4: Prepare data - filter top 10 clothing categories
# ============================================================
import numpy as np

IMAGE_DIR = f"{DATASET_PATH}/images"

# Lấy top 10 category có nhiều ảnh nhất
TOP_N = 10
top_categories = df['articleType'].value_counts().head(TOP_N).index.tolist()
print("Top categories:", top_categories)

# Filter và chỉ giữ ảnh tồn tại
df_filtered = df[df['articleType'].isin(top_categories)].copy()
df_filtered['image_path'] = df_filtered['id'].apply(
    lambda x: f"{IMAGE_DIR}/{x}.jpg"
)
df_filtered = df_filtered[df_filtered['image_path'].apply(os.path.exists)]

# Encode labels
label2idx = {cat: idx for idx, cat in enumerate(top_categories)}
idx2label = {v: k for k, v in label2idx.items()}
df_filtered['label'] = df_filtered['articleType'].map(label2idx)

print(f"\nFiltered samples: {len(df_filtered)}")
print("Label mapping:", label2idx)

# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, cat in enumerate(top_categories):
    sample = df_filtered[df_filtered['articleType'] == cat].iloc[0]
    img = Image.open(sample['image_path']).convert('RGB')
    axes[i//5][i%5].imshow(img)
    axes[i//5][i%5].set_title(cat, fontsize=9)
    axes[i//5][i%5].axis('off')
plt.tight_layout()
plt.savefig('sample_categories.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 5: Dataset & DataLoader
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

class ClothingDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            image = self.transform(image)
        return image, label


# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Split
train_df, val_df = train_test_split(df_filtered, test_size=0.2, 
                                     stratify=df_filtered['label'], random_state=42)

train_dataset = ClothingDataset(train_df, transform=train_transform)
val_dataset   = ClothingDataset(val_df,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
# ============================================================
# CELL 6: Build model - EfficientNet-B0 transfer learning
# ============================================================
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = len(top_categories)
print(f"Device: {DEVICE} | Classes: {NUM_CLASSES}")

# Load pretrained EfficientNet-B0
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Freeze backbone, chỉ train classifier
for param in model.features.parameters():
    param.requires_grad = False

# Thay classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)
print("Model ready.")

# Optimizer & loss
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
# ============================================================
# CELL 7: Training loop
# ============================================================
from tqdm import tqdm

EPOCHS = 15
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, desc='Train' if train else 'Val', leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Epoch [{epoch:02d}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'label2idx': label2idx,
            'idx2label': idx2label,
            'val_acc': val_acc
        }, 'clothing_classifier.pth')
        print(f"  --> Saved best model (val_acc={val_acc:.4f})")

# Unfreeze & fine-tune toàn bộ backbone (phase 2)
print("\n--- Phase 2: Fine-tuning full backbone ---")
for param in model.features.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

for epoch in range(EPOCHS + 1, EPOCHS + 6):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    print(f"Epoch [{epoch:02d}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'label2idx': label2idx,
            'idx2label': idx2label,
            'val_acc': val_acc
        }, 'clothing_classifier.pth')
        print(f"  --> Saved best model (val_acc={val_acc:.4f})")

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# ============================================================
# CELL 8: Evaluation + Confusion Matrix
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
checkpoint = torch.load('clothing_classifier.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(DEVICE))
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=top_categories))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=top_categories, yticklabels=top_categories, cmap='Blues')
plt.title('Confusion Matrix - Clothing Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_clothing.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 9: Plot training history
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'],   label='Val Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'],   label='Val Acc')
ax2.axhline(y=0.8, color='r', linestyle='--', label='Target 80%')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('training_history_clothing.png', dpi=100)
plt.show()
print(f"Model saved: clothing_classifier.pth")
print(f"Best Val Accuracy: {best_val_acc*100:.1f}%")

In [ ]:
# ============================================================
# CELL 10: Inference test
# ============================================================
def predict_clothing(image_path, model, transform, idx2label, device):
    """Predict clothing type from image path."""
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs   = torch.softmax(outputs, dim=1)[0]
        pred_idx = probs.argmax().item()
    return {
        'label': idx2label[pred_idx],
        'confidence': float(probs[pred_idx]),
        'all_scores': {idx2label[i]: float(probs[i]) for i in range(len(idx2label))}
    }

# Test on one sample
test_row = val_df.iloc[0]
result = predict_clothing(test_row['image_path'], model, val_transform, idx2label, DEVICE)
print(f"True label: {idx2label[int(test_row['label'])]}")
print(f"Predicted:  {result['label']} ({result['confidence']*100:.1f}%)")